We want to demonstrate that it is possible to fine-tune a model that is not very capable to complete a fairly challenging task and do quite well. We are going to fine-tune GPT-2 to take textual descriptions of math equations and generate python code for the Mamim library that renders those expressions.

In the process, we will also use a hyperparameter optimization library called optuna that will search for the optimal hyperparameter values within our specified search space.

In [21]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 14.7 MB/s eta 0:00:00


In [1]:
from datasets import load_dataset
dataset = load_dataset("Edoh/manim_python")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/599 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/51 [00:00<?, ? examples/s]

In [ ]:
# Let's look at an example from the training set
#
import random
r = random.randint(0, len(dataset['train'])-1)
print(dataset['train'][r])

{'instruction': 'Rotate the pentagon by 90 degrees counterclockwise over 2 seconds.', 'output': 'from manim import * class MyScene(Scene): def construct(self): pentagon = RegularPolygon(n=5, radius=2, color=ORANGE) self.add(pentagon) self.play(pentagon.animate.rotate(-90 * DEGREES), run_time=2)'}


In [ ]:
# We need to load the model and its corresponding tokenizer. We will use the GPT-2 model for this example.
#
from transformers import GPT2Tokenizer
model_name = "openai-community/gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

We need to prepare the dataset and put it into the format that the model expects it to be in for instruction fine-tuning.
When training a model for Causal Language Modeling (CL), the objective is to predict the next token in a sequence given the previous tokens.

In the Hugging Face transformers implementation (specifically for models like GPT2LMHeadModel), this "shifting" is handled internally by the model during the loss computation.

By setting labels equal to input_ids:

You provide the full ground-truth sequence as the target.
The model internally calculates the loss by comparing its prediction at position i with the label at position i. However, because the model is causal (it can only see tokens 0 to i), this effectively teaches it to predict the next token.

In [10]:
def preprocess_data(examples):
    inputs = [
        f"Instruction: {instr}\nOutput: {out}"
        for instr, out in zip(examples["instruction"], examples["output"])
    ]
    tokenized = tokenizer(inputs, truncation=True, max_length=512, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_datasets = dataset.map(preprocess_data,batched=True,
                                 remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/599 [00:00<?, ? examples/s]

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

In [9]:
# Print out an example
print(tokenized_datasets['train'][0])

{'input_ids': [6310, 2762, 25, 13610, 257, 649, 3715, 3706, 705, 3666, 36542, 4458, 198, 26410, 25, 422, 582, 320, 1330, 1635, 1398, 2011, 36542, 7, 36542, 2599, 825, 5678, 7, 944, 2599, 1208, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50


Optuna is an open-source hyperparameter optimization framework designed to automate the search for optimal hyperparameters in machine learning models. It is framework-agnostic and can be used with PyTorch, TensorFlow, Keras, Scikit-learn, XGBoost, LightGBM, and others.

Here is a summary of its key features:

1. Define-by-Run Style
Unlike traditional frameworks that require a static configuration file, Optuna allows you to construct the search space dynamically using Python code (e.g., loops, conditionals). This makes it highly flexible for complex parameter relationships.

2. Efficient Sampling Algorithms
Optuna uses state-of-the-art algorithms to sample hyperparameters intelligently, often finding better results faster than grid or random search:

TPE (Tree-structured Parzen Estimator): A Bayesian optimization approach (default).
CMA-ES (Covariance Matrix Adaptation Evolution Strategy): Good for continuous hyperparameters.
3. Pruning (Early Stopping)
Optuna can automatically stop unpromising trials early based on intermediate results (e.g., validation loss). This saves computational resources and time.

Median Pruner: Prunes if the trial's intermediate result is worse than the median of previous trials at the same step.
Hyperband Pruner: A bandit-based approach for resource allocation.
4. Easy Integration
It requires minimal code changes. You simply define an objective function that takes a trial object and returns a metric to optimize (like accuracy or loss).

In [11]:
# Define an initializer for the model since we will need to construct it multiple times
# during training and hyperparameter tunning.
#
from transformers import GPT2LMHeadModel

def model_init():
    return GPT2LMHeadModel.from_pretrained(model_name, device_map='auto')

In [12]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./gpt2-manim-python-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    report_to="none",
)

In [14]:
# We need some data to use for evaluation during training. We can create a 
# validation set by splitting the training set.
#
train_val_split = tokenized_datasets["train"].train_test_split(test_size=0.1)
tokenized_datasets["train"] = train_val_split["train"]
tokenized_datasets["validation"] = train_val_split["test"]

In [17]:
from transformers import (
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(model_init=model_init,
                  args=training_args,
                  train_dataset=tokenized_datasets["train"],
                  eval_dataset=tokenized_datasets["validation"],
                  processing_class=tokenizer,
                  data_collator=data_collator,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# These methods are the core mechanism Optuna uses to sample hyperparameters from a defined
# search space. When you call them, Optuna picks a value for that specific trial based on the
# history of previous trials (using its internal sampling algorithm, typically TPE).
#
def hp_space(trial):
    return {
    "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
    "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size",  [2, 4, 8]),
    "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
    "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 6),
    "warmup_steps": trial.suggest_int("warmup_steps", 0, 500),
    "gradient_accumulation_steps": trial.suggest_categorical("gradient_accumulation_steps", [1, 2, 4])
    }

In [ ]:
best_run = trainer.hyperparameter_search(
    direction="minimize",
    backend="optuna",
    n_trials=10,
    hp_space=hp_space,
    compute_objective=lambda metrics: metrics["eval_loss"],
)

[I 2026-02-25 15:27:34,692] A new study created in memory with name: no-name-46b940c7-eec9-431b-b2bd-a99b1c8e38e0
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,0.977247,0.202133
2,0.265100,0.163035
3,0.202179,0.141878
4,0.162321,0.120706
5,0.106987,0.109567


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
[I 2026-02-25 15:46:25,248] Trial 4 finished with value: 0.11582178622484207 and parameters: {'learning_rate': 5.3921898670969184e-05, 'per_device_train_batch_size': 2, 'weight_decay': 0.2766120717016482, 'num_train_epochs': 5, 'warmup_steps': 394, 'gradient_accumulation_steps': 1}. Best is trial 0 with value: 0.10956727713346481.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss


In [ ]:
# Let's see what values it came up with
#
for key, value in best_run.hyperparameters.items():
    print(f"{key}: {value}")

In [ ]:
# We can now pick the best hyperparameters and train a final model using those hyperparameters.
#
for key, value in best_run.hyperparameters.items():
    setattr(training_args, key, value)

trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets.get("validation"),
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)